# Phase 1: Ingest, Clean, Align, and Resample Macro-Economic Data

This notebook guides you through the execution and understanding of the data engineering pipeline. We will:
1. Set up path variables and load credentials.
2. Fetch daily/weekly/monthly macro-economic and sector indicators via FRED, yfinance, and World Bank.
3. Clean and resample everything to month-end frequency.
4. Merge all streams into a master dataset and perform a data quality check.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add parent directory to path to allow importing from the 'src' package
sys.path.insert(0, str(Path.cwd().parent))
from src.data_pipeline import (
    fetch_yfinance_data,
    fetch_fred_data,
    fetch_world_bank_data,
    clean_and_merge,
    generate_quality_report,
    START_DATE,
    END_DATE
)
from src.utils import (
    load_env,
    get_data_paths,
    SECTOR_INDICES,
    MARKET_TICKERS,
    FRED_SERIES,
    WORLD_BANK_INDICATORS
)

env = load_env()
paths = get_data_paths()
print("Environment variables and data paths initialized.")
print(f"Target Date range: {START_DATE} to {END_DATE}")

Environment variables and data paths initialized.
Target Date range: 2009-01-01 to 2026-05-23


### 1. Ingest Market and Equity Data (yfinance)

We fetch Sector Tickers and Market Tickers daily prices, then resample them to monthly.

In [2]:
print("Fetching yfinance data...")
all_yf_tickers = {**SECTOR_INDICES, **MARKET_TICKERS}
equity_df = fetch_yfinance_data(all_yf_tickers, START_DATE, END_DATE)
print(f"yfinance data fetched. Shape: {equity_df.shape}")
equity_df.head()

Fetching yfinance data...
  📥 Downloading Nifty_50 (^NSEI)...
    ✅ 209 monthly observations
  📥 Downloading Nifty_Bank (^NSEBANK)...
    ✅ 209 monthly observations
  📥 Downloading Nifty_IT (^CNXIT)...
    ✅ 209 monthly observations
  📥 Downloading Nifty_Pharma (^CNXPHARMA)...
    ✅ 185 monthly observations
  📥 Downloading Nifty_Auto (^CNXAUTO)...
    ✅ 179 monthly observations
  📥 Downloading Nifty_FMCG (^CNXFMCG)...
    ✅ 185 monthly observations
  📥 Downloading Nifty_Metal (^CNXMETAL)...
    ✅ 179 monthly observations
  📥 Downloading Nifty_Realty (^CNXREALTY)...
    ✅ 191 monthly observations
  📥 Downloading Nifty_Energy (^CNXENERGY)...
    ✅ 185 monthly observations
  📥 Downloading Nifty_Infra (^CNXINFRA)...
    ✅ 191 monthly observations
  📥 Downloading Nifty_PSE (^CNXPSE)...
    ✅ 185 monthly observations
  📥 Downloading Nifty_Media (^CNXMEDIA)...
    ✅ 178 monthly observations
  📥 Downloading India_VIX (^INDIAVIX)...
    ✅ 209 monthly observations
  📥 Downloading USD_INR (USDINR

,Nifty_50,Nifty_Bank,Nifty_IT,Nifty_Pharma,Nifty_Auto,Nifty_FMCG,Nifty_Metal,Nifty_Realty,Nifty_Energy,Nifty_Infra,Nifty_PSE,Nifty_Media,India_VIX,USD_INR,DXY,Global_VIX
Date,,,,,,,,,,,,,,,,
2009-01-31,2874.800049,4456.498047,2225.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.099998,48.733002,86.000000,44.840000
2009-02-28,2763.649902,3892.354736,2094.100098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.610001,50.650002,88.010002,46.349998
2009-03-31,3020.949951,4133.152344,2318.699951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.480000,50.784000,85.430000,44.139999
2009-04-30,3473.949951,5130.890625,2770.850098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.630001,49.188000,84.610001,36.500000
2009-05-31,4448.950195,7414.264160,3191.850098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.299999,46.969002,80.430000,28.920000


### 2. Ingest Macro-Economic Indicators (FRED API)

We fetch global rates, yields, oil, gold, and inflation indices from FRED.

In [3]:
print("Fetching FRED data...")
fred_df = fetch_fred_data(FRED_SERIES, env.get("FRED_API_KEY"), START_DATE, END_DATE)
if not fred_df.empty:
    print(f"FRED data shape: {fred_df.shape}")
    print(fred_df.head())
else:
    print("FRED API Key not set or failed to fetch. Showing empty frame.")

Fetching FRED data...
  📥 Downloading US_Fed_Funds_Rate (FEDFUNDS) from FRED...
    ✅ 208 monthly observations
  📥 Downloading US_CPI (CPIAUCSL) from FRED...
    ✅ 208 monthly observations
  📥 Downloading US_10Y_Yield (DGS10) from FRED...
    ✅ 209 monthly observations
  📥 Downloading US_GDP (GDP) from FRED...
    ✅ 205 monthly observations
  📥 Downloading Brent_Crude (DCOILBRENTEU) from FRED...
    ✅ 209 monthly observations
  📥 Downloading Gold_Price (GOLDAMGBD228NLBM) from FRED...
    ❌ Failed: Bad Request.  The series does not exist.

  ⚠️  Failed FRED series: Gold_Price
FRED data shape: (209, 5)
            US_Fed_Funds_Rate   US_CPI  US_10Y_Yield     US_GDP  Brent_Crude
Date                                                                        
2009-01-31               0.15  211.933      2.517500  14430.902    43.439500
2009-02-28               0.22  212.705      2.870000  14430.902    43.324737
2009-03-31               0.18  212.495      2.819545  14430.902    46.540455
2009-04

### 3. Ingest Macro Indicators (World Bank API)

We fetch annual indicators for India (GDP, inflation) and resample/interpolate them.

In [4]:
print("Fetching World Bank data...")
wb_df = fetch_world_bank_data(WORLD_BANK_INDICATORS)
if not wb_df.empty:
    print(f"World Bank data shape: {wb_df.shape}")
    print(wb_df.head())
else:
    print("Failed to fetch World Bank data.")

Fetching World Bank data...
  📥 Downloading India_GDP (NY.GDP.MKTP.CD) from World Bank...
    ✅ 193 monthly observations (from 17 annual)
  📥 Downloading India_CPI (FP.CPI.TOTL.ZG) from World Bank...
    ✅ 193 monthly observations (from 17 annual)
World Bank data shape: (193, 2)
               India_GDP  India_CPI
Date                               
2009-12-31  1.341888e+12  10.882353
2010-01-31  1.341888e+12  10.882353
2010-02-28  1.341888e+12  10.882353
2010-03-31  1.341888e+12  10.882353
2010-04-30  1.341888e+12  10.882353


### 4. Clean, Merge, and Build the Master Dataset

Align all dates, forward-fill minor gaps, compute sector returns, and compile the master database.

In [5]:
print("Merging datasets and computing returns...")
master = clean_and_merge(equity_df, fred_df, wb_df)
master_path = paths["processed"] / "master_dataset.csv"
master.to_csv(master_path)
print(f"Master dataset successfully created and saved at: {master_path}")
print(f"Shape: {master.shape}")

# Generate the Quality Report too
generate_quality_report(master, paths["processed"] / "data_quality_report.txt")

Merging datasets and computing returns...
Master dataset successfully created and saved at: /Users/neil/Desktop/Macro/data/processed/master_dataset.csv
Shape: (209, 35)
  📄 Quality report saved: /Users/neil/Desktop/Macro/data/processed/data_quality_report.txt
